In [18]:
######################################################################################################
# ALUNO : Daniel Marques Diniz Maia
# MATRÍCULA : 2026061836
# TURMA : TM
# MÉTODOS DE BUSCA GLOBAL
######################################################################################################

In [19]:
import numpy as np
import random
from scipy.optimize import minimize, differential_evolution, dual_annealing


######################################################################################################

# Definindo as funções de busca

######################################################################################################

## Particle Swarm

Modelo-base foi retirado do Teams e modificado para facilitar o uso aqui

In [20]:
class Particle:

    def __init__(self, bounds, func):
        (x_min, x_max),(y_min, y_max) = bounds

        # bounds e func são argumentos passados na chamada, que vão mudar a cada exercício
        self.func = func
        self.x_min, self.x_max = x_min, x_max
        self.y_min, self.y_max = y_min, y_max

        self.position = np.array([
            np.random.uniform(x_min, x_max),
            np.random.uniform(y_min, y_max)
        ])

        self.velocity = np.random.uniform(-1, 1, 2)

        self.best_position = self.position.copy()
        self.best_value = self.func(self.position[0], self.position[1])

    def evaluate(self):
        value = self.func(self.position[0], self.position[1])

        if value < self.best_value:
            self.best_value = value
            self.best_position = self.position.copy()

        return value

    def update_velocity(self, global_best):
        r1 = np.random.rand(2)
        r2 = np.random.rand(2)

        cognitive = 1.5 * r1 * (self.best_position - self.position) # C1 = 1.5, C2 = 1.5, W=0.7
        social = 1.5 * r2 * (global_best - self.position)

        self.velocity = 0.7 * self.velocity + cognitive + social

    def update_position(self):
        self.position += self.velocity

        self.position[0] = np.clip(self.position[0], self.x_min, self.x_max)
        self.position[1] = np.clip(self.position[1], self.y_min, self.y_max)


def pso(bounds, func):
    """
    bounds = (x_min, x_max, y_min, y_max)
    func   = função f(x, y) a ser minimizada
    """
    swarm = [Particle(bounds, func)
             for _ in range(30)] # num_particles = 30

    global_best_position = None
    global_best_value = float("inf")

    for _ in range(100): # max_iterations = 100

        for particle in swarm:
            value = particle.evaluate()

            if value < global_best_value:
                global_best_value = value
                global_best_position = particle.position.copy()

        for particle in swarm:
            particle.update_velocity(global_best_position)
            particle.update_position()

    return global_best_position, global_best_value

## Algoritmo genético binário

Modelo-base foi retirado do Teams e modificado para facilitar o uso aqui

In [21]:
# =====================================
# PARÂMETROS
# =====================================

# POP_SIZE = 10
# GENERATIONS = 150
# BITS_PER_VAR = 16
# CROSSOVER_RATE = 0.8
# MUTATION_RATE = 0.01

# =====================================
# CREATE CHROMOSOME
# =====================================

def random_chromosome():
    return ''.join(random.choice('01')
                   for _ in range(64)) # CHROM_LENGTH = 2 * BITS_PER_VAR (32)


# =====================================
# DECODE CHROMOSOME
# =====================================

def decode(chromosome,bounds):

    (xmin,xmax),(ymin,ymax) = bounds

    x_bits = chromosome[:32] # BITS_PER_VAR = 32
    y_bits = chromosome[32:]

    x_int = int(x_bits, 2)
    y_int = int(y_bits, 2)

    max_int = 2**32 - 1

    x = xmin + (x_int/max_int)*(xmax-xmin)
    y = ymin + (y_int/max_int)*(ymax-ymin)

    return x, y


# =====================================
# FITNESS
# Convert minimization into maximization
# =====================================

def fitness(chromosome, funcao,bounds):

    x, y = decode(chromosome,bounds)

    f = funcao(x, y)

    return 1/(1 + f)


# =====================================
# INITIAL POPULATION
# =====================================

def initialize_population():

    return [random_chromosome()
            for _ in range(100)] # POP_SIZE = 100


# =====================================
# ROULETTE SELECTION (otimizada)
# =====================================

def selection(population, fitness_values): # fitness_values será definido em genetic_algorithm
    total = sum(fitness_values)
    r = random.uniform(0, total)
    cumulative = 0

    for chromosome, fit in zip(population, fitness_values):
        cumulative += fit
        if cumulative >= r:
            return chromosome

    return population[-1]


# =====================================
# SINGLE-POINT CROSSOVER
# =====================================

def crossover(parent1, parent2):

    if random.random() < 0.8: # CROSSOVER_RATE = 0.8

        point = random.randint(1,63) # CRHOMOSOME_LENGTH-1 = 64 -1

        child1 = parent1[:point] + parent2[point:]
        child2 = parent2[:point] + parent1[point:]

        return child1, child2

    return parent1, parent2


# =====================================
# MUTATION
# =====================================

def mutation(chromosome):

    chromosome = list(chromosome)

    for i in range(len(chromosome)):

        if random.random() < 0.07: # MUTATION_RATE = 0.07

            chromosome[i] = (
                '0'
                if chromosome[i] == '1'
                else '1'
            )

    return ''.join(chromosome)


# =====================================
# GENETIC ALGORITHM
# =====================================

def genetic_algorithm(funcao, bounds):

    population = initialize_population()

    best_solution = None
    best_objective = float('inf')

    for generation in range(150):

        # Avalia a população uma única vez por geração
        values = [funcao(*decode(c, bounds)) for c in population]

        for chromosome, f in zip(population, values):
            if f < best_objective:
                best_objective = f
                best_solution = chromosome

        # Correção no fitness: na versão original, teríamos problemas com valores negativos de f(x,y). Queremos ter mais chance De selecionar o melhor inivíduo (menor f), então
        # o pior será f_max. A diferença dele para um dado ponto é grande quando esse ponto tem um valor muito pequeno, que é o que buscamos, e queremos selecionar. Então,
        # o mais lógico é selecionar os indivíduos que tem a maior diferença para f_max. Adicionar 1e-12 é uma proteção para valores de f todos iguais, o que anularia as
        # probabilidades e quebraria a roleta
        f_max = max(values)
        fitness_values = [f_max - f + 1e-12 for f in values]

        new_population = []
        while len(new_population) < 100:
            parent1 = selection(population, fitness_values)
            parent2 = selection(population, fitness_values)

            child1, child2 = crossover(parent1, parent2)

            new_population.extend([mutation(child1), mutation(child2)])

        population = new_population[:100]

    return best_solution, best_objective

## Mão na massa

In [24]:
def minimo_global(bounds, funcao):
    '''
    Função Python que recebe a lista [(x_min,x_max),(y_min,y_max)], a função que queremos minimizar e imprime os valores encontrados para x,y e f(x,y) por todos os métodos.
    '''

    vfunc = lambda v: funcao(*v) # Powell precisa de uma função vetorial

    # PARTICLE SWARM
    posicaop, fminp = pso(bounds, funcao)
    resp = minimize(vfunc, posicaop, method='Powell') 
    print(f'PARTICLE SWARM (polido com Powell) - xmin = {resp.x[0]:.5e}; ymin = {resp.x[1]:.5e} /// f(xmin) = {resp.fun:.5e}')

    # DIFFERENTIAL EVOLUTION
    resd = differential_evolution(vfunc, bounds,maxiter=100, popsize=20,disp=False, polish=True)
    print(f'DIFFERENTIAL EVOLUTION (polido com L-BFGS-B) - xmin = {resd.x[0]:.5e}; ymin = {resd.x[1]:.5e} /// f(xmin) = {resd.fun:.5e}')

    # ALGORITMO GENÉTICO BINÁRIO
    cromossomo = genetic_algorithm(funcao, bounds)[0]
    posicaog = decode(cromossomo, bounds)      # (x, y)
    resg = minimize(vfunc, posicaog, method='Powell') 
    print(f'ALGORITMO GENÉTICO BINÁRIO (polido com Powell) - xmin = {resg.x[0]:.5e}; ymin = {resg.x[1]:.5e} /// f(xmin) = {resg.fun:.5e}')

    # SIMULATED ANNEALING
    # dual_annealing com no_local_search=True é um Generalized Simulated Annealing (GSA), que dá um tratamento mais sofisticado que o SA padrão: 
    # usa distribuição de visita de Tsallis na geração de candidatos e na aceitação dos valores, além de reiniciar a temperatura quando fica muito baixa
    posicaodual = dual_annealing(vfunc, bounds, maxiter=1000, seed=None, no_local_search=True).x # retorna (x,y), fmin. Precisamos só de (x,y) para polir com Powell
    ressa = minimize(vfunc, posicaodual, method='Powell') 
    print(f'GENERALIZED SIMULATED ANNEALING (polido com Powell) - xmin = {ressa.x[0]:.5e}; ymin = {ressa.x[1]:.5e} /// f(xmin) = {ressa.fun:.5e}')

# Um dicionário com as funções simplifica o código

import numpy as np

dic_funcoes = {
    "1": lambda x, y: -10*np.exp(-((x-5)**2 + (y-5)**2)/4)
                      - 4*np.exp(-((x+6)**2 + (y+4)**2)/4),

    "2": lambda x, y: -15*np.exp(-((x-8)**2 + (y-2)**2)/9)
                      - 6*np.exp(-((x+8)**2 + (y+3)**2)/9),

    "3": lambda x, y: (x**2 + y**2)/100
                      - 12*np.exp(-((x-4)**2 + (y-7)**2)/2)
                      - 5*np.exp(-((x+7)**2 + (y-5)**2)/2),

    "4": lambda x, y: -20*np.exp(-((x-10)**2 + (y-10)**2)/16)
                      - 8*np.exp(-((x+10)**2 + (y+10)**2)/16),

    "5": lambda x, y: (x**2 + y**2)/200
                      - 18*np.exp(-((x-6)**2 + (y+6)**2))
                      - 7*np.exp(-((x+5)**2 + (y-5)**2)),

    "6": lambda x, y: -14*np.exp(-((x-12)**2 + (y+2)**2)/3)
                      - 9*np.exp(-((x+3)**2 + (y-12)**2)/3),

    "7": lambda x, y: 0.01*(x**2 + y**2)
                      - 25*np.exp(-((x-7)**2 + (y-7)**2)/5)
                      - 10*np.exp(-((x+8)**2 + (y+8)**2)/5),

    "8": lambda x, y: -16/(1 + 0.5*((x-5)**2 + (y+10)**2))
                      - 5/(1 + 0.5*((x+10)**2 + (y-3)**2)),

    "9": lambda x, y: -13*np.exp(-((x-15)**2 + (y-1)**2)/4)
                      - 3*np.exp(-((x+15)**2 + (y+1)**2)/4),

    "10": lambda x, y: (x**2 + y**2)**2/1e5
                       - 17*np.exp(-((x-9)**2 + (y-4)**2)/6)
                       - 8*np.exp(-((x+4)**2 + (y+9)**2)/6),
}


intervalo = [(-25,25),(-25,25)]
for f in dic_funcoes:
    print('='*60)
    print(f'QUESTÃO {f}\n')
    minimo_global(intervalo,dic_funcoes[f])

QUESTÃO 1

PARTICLE SWARM (polido com Powell) - xmin = 5.00000e+00; ymin = 5.00000e+00 /// f(xmin) = -1.00000e+01
DIFFERENTIAL EVOLUTION (polido com L-BFGS-B) - xmin = 5.00000e+00; ymin = 5.00000e+00 /// f(xmin) = -1.00000e+01
ALGORITMO GENÉTICO BINÁRIO (polido com Powell) - xmin = 5.00000e+00; ymin = 5.00000e+00 /// f(xmin) = -1.00000e+01
GENERALIZED SIMULATED ANNEALING (polido com Powell) - xmin = 5.00000e+00; ymin = 5.00000e+00 /// f(xmin) = -1.00000e+01
QUESTÃO 2

PARTICLE SWARM (polido com Powell) - xmin = 8.00000e+00; ymin = 2.00000e+00 /// f(xmin) = -1.50000e+01
DIFFERENTIAL EVOLUTION (polido com L-BFGS-B) - xmin = 8.00000e+00; ymin = 2.00000e+00 /// f(xmin) = -1.50000e+01
ALGORITMO GENÉTICO BINÁRIO (polido com Powell) - xmin = 7.99995e+00; ymin = 1.99997e+00 /// f(xmin) = -1.50000e+01
GENERALIZED SIMULATED ANNEALING (polido com Powell) - xmin = 8.00000e+00; ymin = 2.00000e+00 /// f(xmin) = -1.50000e+01
QUESTÃO 3

PARTICLE SWARM (polido com Powell) - xmin = 3.99334e+00; ymin = 6